# Chapter 00 — GPU Benchmark: FP32 vs FP16 vs BF16 Matmul

Large language models are, at their core, compute-bound workloads. Before writing a single line of model code, you need to understand the hardware that will execute it. This chapter is the true prerequisite: it maps the physical infrastructure — GPUs, memory, interconnects, and cloud clusters — that determines whether your training run takes hours or weeks.

In [ ]:
import torch
import subprocess
import time

def parse_nvidia_smi():
    """Parse nvidia-smi to get GPU info."""
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,temperature.gpu",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True
    )
    for i, line in enumerate(result.stdout.strip().split("\n")):
        name, mem_total, mem_free, temp = line.split(", ")
        print(f"GPU {i}: {name} | {mem_total} MB total | {mem_free} MB free | {temp}°C")

def benchmark_matmul(size=4096, dtype=torch.float32, warmup=10, iters=100):
    """Benchmark matrix multiplication at a given precision."""
    device = torch.device("cuda")
    a = torch.randn(size, size, dtype=dtype, device=device)
    b = torch.randn(size, size, dtype=dtype, device=device)

    # Warmup
    for _ in range(warmup):
        torch.mm(a, b)
    torch.cuda.synchronize()

    # Timed iterations
    start = time.perf_counter()
    for _ in range(iters):
        torch.mm(a, b)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    tflops = (2 * size**3 * iters) / (elapsed * 1e12)
    mem_mb = torch.cuda.max_memory_allocated() / 1e6
    print(f"{str(dtype):20s} | {elapsed/iters*1000:8.2f} ms/iter | {tflops:6.1f} TFLOPS | {mem_mb:8.1f} MB peak")

if __name__ == "__main__":
    parse_nvidia_smi()
    print("\n--- Matrix Multiplication Benchmark (4096x4096) ---")
    print(f"{'Dtype':20s} | {'Time':>12s} | {'TFLOPS':>10s} | {'Memory':>12s}")
    print("-" * 65)
    for dtype in [torch.float32, torch.float16, torch.bfloat16]:
        torch.cuda.reset_peak_memory_stats()
        benchmark_matmul(dtype=dtype)

---

**Course:** Aprenda LLM | **Chapter 00**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.